
Pipeline de Dados - Camada Silver
Consolidação dos Dados de Atividades Pesqueiras
Este notebook unifica os dados das tabelas da camada Bronze (`fixed_gear`, `pole_and_line` e `purse_seines`), aplica filtros de alta acurácia para atividade de pesca (`is_fishing > 0.7`) e enriquece os registros identificando o tipo de pesca utilizada e a data do evento.



In [0]:
import logging
from delta.tables import DeltaTable
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

In [0]:
# Configuração de Logging
logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger("silver_pipeline")

In [0]:
CATALOG = 'fishing'
BRONZE  = 'bronze'
SILVER  = 'silver'

In [0]:
# Nome das Tabelas de Origem (Bronze)
FIXED_GEAR_TABLE_NAME    = f'{CATALOG}.{BRONZE}.fixed_gear'
POLE_AND_LINE_TABLE_NAME = f'{CATALOG}.{BRONZE}.pole_and_line'
PURSE_SEINES_TABLE_NAME  = f'{CATALOG}.{BRONZE}.purse_seines'

In [0]:
SILVER_FISHING_TABLE_NAME = f'{CATALOG}.{SILVER}.consolidated_fishing_activity'

In [0]:
# is fishing vai de 0 a 1, quanto mais perto de 1 maior a acurácia. Por isso, peguei valores acima de 0.7
FISHING_THRESHOLD = 0.7

In [0]:
# MAGIC %md
# MAGIC ## 2. Função de Ingestão (Upsert via MERGE)

# COMMAND ----------

def merge_to_delta(df: DataFrame, table_name: str, merge_keys: list[str] = ["mmsi", "timestamp", "gear_type"]) -> None:
    """
    Realiza o Upsert (Merge) do DataFrame PySpark na tabela Delta Target.
    Cria a tabela caso ela ainda não exista no Unity Catalog.
    """
    if not spark.catalog.tableExists(table_name):
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .saveAsTable(table_name)
        LOGGER.info(f"Tabela Delta '{table_name}' criada com sucesso.")
        return

    join_condition = " AND ".join([f"target.{key} = source.{key}" for key in merge_keys])
    delta_table = DeltaTable.forName(spark, table_name)

    (
        delta_table.alias('target')
        .merge(
            df.alias('source'),
            join_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    LOGGER.info(f"MERGE concluído com sucesso na tabela '{table_name}'.")

In [0]:
LOGGER.info("Iniciando leitura das tabelas da camada Bronze...")

# Leitura e inclusão do campo 'gear_type' (Tipo de Pesca)
df_fixed = (
    spark.table(FIXED_GEAR_TABLE_NAME)
    .withColumn("gear_type", F.lit("Fixed Gear"))
)

df_pole = (
    spark.table(POLE_AND_LINE_TABLE_NAME)
    .withColumn("gear_type", F.lit("Pole and Line"))
)

df_purse = (
    spark.table(PURSE_SEINES_TABLE_NAME)
    .withColumn("gear_type", F.lit("Purse Seines"))
)

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Consolidação e Aplicação das Regras de Negócio

# COMMAND ----------

LOGGER.info("Unificando tabelas e aplicando transformações da camada Silver...")

# 1. União de todas as tabelas em um único DataFrame
df_unified = df_fixed.unionByName(df_pole).unionByName(df_purse)

# 2. Aplicação das Regras de Negócio:
#    - Filtro de alta acurácia (is_fishing > 0.7)
#    - Extração da data em que aconteceu a pesca (fishing_date)
#    - Seleção e organização do schema final
df_silver = (
    df_unified
    .filter(F.col("is_fishing") > FISHING_THRESHOLD)
    .withColumn("fishing_date", F.to_date(F.col("timestamp")))
    .select(
        F.col("mmsi"),
        F.col("timestamp"),
        F.col("fishing_date"),
        F.col("latitude"),
        F.col("longitude"),
        F.col("gear_type"),
        F.col("is_fishing").alias("fishing_probability")
    )
)

# Exibe prévia da base consolidada
display(df_silver)

In [0]:
# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Salva a Tabela Consolidada na Camada Silver

# COMMAND ----------

LOGGER.info(f"Salvando base consolidada na camada Silver: {SILVER_FISHING_TABLE_NAME}")

# Realiza o MERGE/Upsert usando a chave composta (mmsi + timestamp + gear_type)
merge_to_delta(
    df=df_silver,
    table_name=SILVER_FISHING_TABLE_NAME,
    merge_keys=["mmsi", "timestamp", "gear_type"]
)

LOGGER.info("Consolidação da camada Silver concluída com sucesso! 🚀")

Exemplos de análise na base consolidada: <br>
Quais regiões tem maior quantidade de atividades pesqueiras?


In [0]:
df_result = (
    spark.table("fishing.silver.consolidated_fishing_activity")
    .filter(F.col("fishing_probability") >= 1.0)
    .groupBy(
        F.round(F.col("latitude"), 1).alias("region_latitude"),
        F.round(F.col("longitude"), 1).alias("region_longitude")
    )
    .agg(F.count("*").alias("fishing_activity_count"))
    .orderBy(F.col("fishing_activity_count").desc())
    .limit(10)
)

display(df_result)



Exemplos de análise na base consolidada: <br>
Considerando que em fevereiro de 2015 a atividade com rede de cerco era limitada.
Quais navios estiveram pescando com rede de cerco em fevereiro de 2015?


In [0]:
df_result = (
    spark.table("fishing.silver.consolidated_fishing_activity")
    .filter(
        (F.col("fishing_probability") == 1.0) &
        (F.col("gear_type") == "Purse Seines") &
        (F.col("fishing_date") >= F.to_date(F.lit("2015-02-01"))) &
        (F.col("fishing_date") <= F.to_date(F.lit("2015-02-28")))
    )
    .agg(
        F.countDistinct("mmsi").alias("vessels_definitely_fishing"),
        F.collect_set("mmsi").alias("mmsi_list")
    )
)

display(df_result)